# Chapter 12 &mdash; Designing Practical PDA in Markdown: $L_{abORac}$

**Concept 8 of the Chapter 12 decomposition:** *Designing Practical PDA in Markdown: $L_{abORac}$*

$\{a^ib^jc^k : i=j$ or $i=k\}$ &mdash; designed nondeterministically, with every line commented.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-Designing-PDA-In-Markdown/Concept-Designing-PDA-In-Markdown.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.AnimatePDA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimatePDA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimatePDA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


The design method for a nondeterministic PDA:

1. **name the guesses.** Here there are two: "the $b$s will match" and "the $c$s will
   match". Each becomes an $\varepsilon$ branch out of the counting state.
2. **count the $a$s** onto the stack before branching.
3. **write one arm per guess**, and let the wrong guess die.
4. **comment every line** &mdash; a PDA table is unreadable otherwise.

The union structure of the language maps directly onto the branch structure of the
machine, exactly as it did for the grammar in Chapter 11, Concept 11.

## 2. Definitions

### The machine, every line commented

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

In [ ]:
AbOrAc = md2mc('''PDA
!! L = { a^i b^j c^k : i = j  or  i = k }
!! --- count the a's -------------------------------------------------
I  : a , # ; A#  -> I      !! first a: push a marker above #
I  : a , A ; AA  -> I      !! each further a: one more marker
!! --- guess 1: the b's will match the a's ---------------------------
I  : '' , # ; #  -> B      !! guess with zero a's counted
I  : '' , A ; A  -> B      !! guess with some a's counted
B  : b , A ; ''  -> B      !! each b cancels one a
B  : '' , # ; #  -> BC     !! all a's cancelled: the c's are unconstrained
BC : c , # ; #   -> BC     !! consume any number of c's
BC : '' , # ; #  -> F
!! --- guess 2: the c's will match the a's ---------------------------
I  : '' , # ; #  -> C
I  : '' , A ; A  -> C
C  : b , A ; A   -> C      !! skip the b's WITHOUT touching the count
C  : b , # ; #   -> C      !! ... including when i = 0, so the stack is just #
C  : '' , A ; A  -> CC
C  : '' , # ; #  -> CC
CC : c , A ; ''  -> CC     !! each c cancels one a
CC : '' , # ; #  -> F
''')

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

### The specification

In [ ]:
def in_abac(s):
    i = len(s) - len(s.lstrip('a')); rest = s[i:]
    j = len(rest) - len(rest.lstrip('b')); k = len(rest) - j
    return s == 'a'*i + 'b'*j + 'c'*k and (i == j or i == k)

<!-- nav-strip -->

---

&larr;&nbsp;[Ch12&nbsp;7.&nbsp;PDA Behavior Through Examples, Stack Limits, and Loop Prevention](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-Stack-Limits-And-Loops/Concept-Stack-Limits-And-Loops.ipynb) &nbsp;&middot;&nbsp; [**Chapter 12** index](https://github.com/ganeshutah/Jove/blob/master/Chapter12-PDA/README.md) &nbsp;&middot;&nbsp; [Ch12&nbsp;9.&nbsp;CFG to PDA Conversion: the Goal/Subgoal Stack Machine](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-CFG-To-PDA/Concept-CFG-To-PDA.ipynb)&nbsp;&rarr;

---

## 3. Tests

The two guesses are the two $\varepsilon$ branches out of `I`.

In [ ]:
branches = sorted({(k, q2) for k, v in AbOrAc["Delta"].items()
                   for (q2, _) in v if k[0] == 'I' and k[1] == ''})
for b in branches: print("  ", b)
assert {q for _, q in branches} == {'B', 'C'}

Each arm handles one disjunct; the wrong guess dies.

In [ ]:
for s, why in [('aabb', 'i=j only'), ('aacc', 'i=k only'),
               ('abc', 'both'), ('aabbc', 'i=j, c free')]:
    print("  %-8r (%-10s) accepted %-6s computations %d"
          % (s, why, pda_accepts(AbOrAc, s, STKMAX=8),
             pda_npaths(AbOrAc, s, STKMAX=8)))
assert pda_accepts(AbOrAc, 'aabb', STKMAX=8)
assert pda_accepts(AbOrAc, 'aacc', STKMAX=8)

Strings satisfying **neither** disjunct are rejected.

In [ ]:
for s in ['aabc', 'abbcc', 'aaabbc']:
    print("  %-8r in L? %-6s PDA %s" % (s, in_abac(s), pda_accepts(AbOrAc, s, STKMAX=8)))
    assert not in_abac(s) and not pda_accepts(AbOrAc, s, STKMAX=8)
print("\n('abbc' looks like a non-member but has i = k = 1, so it IS in L.)")

Exhaustive agreement with the specification.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(5) for p in product('abc', repeat=k)]
bad = [s for s in strs if pda_accepts(AbOrAc, s, STKMAX=6) != in_abac(s)]
print("mismatches over %d strings :" % len(strs), bad)
assert not bad

Note the `C` arm **peeks** at the b's rather than popping &mdash; the design subtlety.

In [ ]:
print("C : b , A ; A -> C      pop A, push A back -- a PEEK")
print()
print("If it popped, the b's would eat the a-count that the c's still need.")
print("This is the line beginners get wrong, and the comment is why you find it.")

## 4. Animation

The two-armed machine; the fork out of `I` is the guess.

In [ ]:
from jove.AnimatePDA import *
AnimatePDA(AbOrAc, FuseEdges=True)

## 5. Exercises


1. Add a third disjunct, $j=k$. Where does the new arm attach?
2. Why must the guess be made **after** counting the $a$s?
3. Could a deterministic PDA recognise this language? (Chapter 11, Concept 17.)

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 253 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter12-PDA/Concept-Designing-PDA-In-Markdown')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')